In [5]:
%load_ext autoreload
%reload_ext autoreload
%autoreload 2

import config as cfg
import os

import torch
from torch import nn
from torch.utils.data import DataLoader

from Data.data_loading import load_and_preprocess_data, create_tensor_from_dataframe, create_sequences, create_dataloaders 
from Training.train_matt import Trainer
from Training.basicEval import plotLoss, plotAccuracy, reportFinalMetrics, reportMultiFinalMetrics, plotMultiAccuracy, plotMultiLoss
from Model.model_split import FrameTransformer, print_model_info

from Training.customLoss import ADELoss, FDELoss, RMSELoss

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
root_dir = os.getcwd()  # Use current working directory as root
data_dir = os.path.join(root_dir, 'Data')
csv_dir = os.path.join(data_dir, 'one_csv')
csv_file = os.path.join(csv_dir, 'michael_10s_processed.csv')

print("Data directory: ", data_dir)
print("CSV directory: ", csv_dir)
print("CSV file: ", csv_file)


model_dir = os.path.join(root_dir, 'Model')
save_model_dir = os.path.join(model_dir, 'Saved_Model')
print("Model directory: ", model_dir)
print("Saved model directory: ", save_model_dir)


Data directory:  /home/jaskin/Deep-Learning-Project/Data
CSV directory:  /home/jaskin/Deep-Learning-Project/Data/one_csv
CSV file:  /home/jaskin/Deep-Learning-Project/Data/one_csv/michael_10s_processed.csv
Model directory:  /home/jaskin/Deep-Learning-Project/Model
Saved model directory:  /home/jaskin/Deep-Learning-Project/Model/Saved_Model


In [ ]:
import numpy as np

def test_model(model, test_loader, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set the model to evaluation mode
    
    all_predictions = []
    all_targets = []
    total_loss = 0.0
    loss_fn = torch.nn.MSELoss()

    with torch.no_grad():  # Disable gradient computation for faster testing
        for inputs, targets in test_loader:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)  # Make predictions on the batch
            
            batch_loss = loss_fn(outputs, targets)
            total_loss += batch_loss.item() * inputs.size(0) # Weighted by batch size
            
            all_predictions.append(outputs.cpu().numpy())
            all_targets.append(targets.cpu().numpy())

    # Concatenate all predictions and targets from batches
    all_predictions = np.concatenate(all_predictions, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)
    
    # Calculate overall metrics
    avg_loss = total_loss / len(test_loader.dataset)
    print(f'Test Loss (MSE): {avg_loss:.6f}')

    # Reshape for metric calculation if necessary (assuming [batch, pred_len, num_ids, 2])
    # Adjust axis based on your actual output shape and how ADE/RMSE should be computed
    if len(all_predictions.shape) == 4:
        # Example: Calculate error per point across batch, pred_len, num_ids
        errors = np.linalg.norm(all_predictions - all_targets, axis=-1) # Norm along the last axis (X, Y)
        ade = np.mean(errors)
        rmse = np.sqrt(np.mean((all_predictions - all_targets) ** 2))
    else:
        # Fallback for simpler shapes or adjust as needed
        displacement_errors = np.linalg.norm(all_predictions - all_targets, axis=-1)
        ade = np.mean(displacement_errors)
        rmse = np.sqrt(np.mean((all_predictions - all_targets) ** 2))
        
    print(f'Average Displacement Error (ADE): {ade:.6f}')
    print(f'Root Mean Squared Error (RMSE): {rmse:.6f}')

    # Display predicted vs actual for the first 5 examples (first sequence, first ID, first prediction step)
    print("\nPredicted vs Actual (First 5 examples - first ID, first pred step):")
    if len(all_predictions.shape) == 4:
        print("Predicted:", all_predictions[:5, 0, 0, :])
        print("Actual:", all_targets[:5, 0, 0, :])
    else:
        print("Predicted:", all_predictions[:5])
        print("Actual:", all_targets[:5])
    return all_predictions


df, transformer_max_ids_per_frame, frame_scaler = load_and_preprocess_data(csv_folder=csv_dir)

# 2. Create tensor from dataframe
all_data_tensor = create_tensor_from_dataframe(df, transformer_max_ids_per_frame)

# 3. Create input-output sequences
X, Y = create_sequences(all_data_tensor)

# 4. Create dataloaders for training and testing
train_loader, test_loader, train_prefetcher, test_prefetcher = create_dataloaders(X, Y)

model = FrameTransformer(
        input_feature_size=cfg.NUM_INPUT_FEATURES, 
        num_ids=X.size(2), 
        sequence_length=X.size(1),  
        prediction_length=cfg.PREDICTION_LENGTH,
        hidden_size=64,  
        num_heads=cfg.NUM_HEADS,
        dropout_rate=cfg.DROPOUT_RATE
)

trainScript = Trainer(model, train_loader, test_loader)

trainScript.earlyStop(enable=True, patience=30, delta=0.01)
train_losses1, val_losses1, train_accs1, val_accs1, epoch_times1 = trainScript.train(
    num_epochs=cfg.EPOCHS, 
    learningRate=cfg.LEARNING_RATE, 
    criterion=nn.MSELoss(), 
    optimizer=torch.optim.Adam(model.parameters(), lr=cfg.LEARNING_RATE)
)



All CSVs now have 297 frames after trimming
Minimum records per ID: 1
Average records per ID: 202.94
Maximum records per ID: 289

Minimum IDs (Vehicles) per frame: 9
Average IDs (Vehicles) per frame: 12.30
Maximum IDs (Vehicles) per frame: 16

After normalization:
X range: 0.0000 to 5.0000
Y range: 0.0000 to 5.0000
Height range: 0.0000 to 5.0000
Width range: 0.0000 to 5.0000
Frame range: 0.0000 to 5.0000
Determined tensor ID dimension size based on max(ID_Norm): 18
All data tensor shape: torch.Size([1, 297, 18, 5])
All data tensor shape: torch.Size([1, 297, 18, 5])

Training on device: cuda

Training on device: cuda


Training Progress:   0%|          | 0/50 [00:00<?, ?it/s]

Epoch 1/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 1/50 [Val]:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 2/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

...

Epoch 49/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 49/50 [Val]:   0%|          | 0/2 [00:00<?, ?it/s]

Epoch 50/50 [Train]:   0%|          | 0/5 [00:00<?, ?it/s]

Epoch 50/50 [Val]:   0%|          | 0/2 [00:00<?, ?it/s]


Training complete in 13.94 seconds, or 0.23 minutes
Total epochs run: 50
Average time per epoch: 0.28 seconds
Inference time per batch: 0.06 seconds
Final Training Loss: 0.6888
Final Validation Loss: 0.7293
Final Training Accuracy: 89.51%
Final Validation Accuracy: 89.94%


In [11]:
import csv
from tqdm.notebook import tqdm

def export_predictions_to_csv(model, data_loader, csv_path, device=None):
    """Exports model predictions to a CSV file."""
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    headers = ['sequence_index', 'prediction_step', 'id_index', 'pred_x', 'pred_y']

    print(f"Exporting predictions to {csv_path}...")
    with open(csv_path, 'w', newline='') as csvfile:
        csv_writer = csv.writer(csvfile)
        csv_writer.writerow(headers)
        
        global_sequence_index = 0
        with torch.no_grad():
            for batch_idx, (inputs, targets) in enumerate(tqdm(data_loader, desc="Exporting")):
                inputs = inputs.to(device)
                targets = targets.to(device) # Keep targets for checking padding
                
                outputs = model(inputs)
                
                batch_size, pred_len, num_ids, _ = outputs.shape
                
                outputs_np = outputs.cpu().numpy()
                targets_np = targets.cpu().numpy() # For easier padding check
                
                for i in range(batch_size):
                    current_sequence_index = global_sequence_index + i
                    for t in range(pred_len):
                        for id_idx in range(num_ids):
                            # Check if the target data indicates padding for this point
                            if targets_np[i, t, id_idx, 0] != cfg.PADDING_TOKEN:
                                pred_point = outputs_np[i, t, id_idx]
                                row = [
                                    current_sequence_index,
                                    t,
                                    id_idx,
                                    f"{pred_point[0]:.6f}",
                                    f"{pred_point[1]:.6f}"
                                ]
                                csv_writer.writerow(row)
                                
                global_sequence_index += batch_size
                
    print(f"Finished exporting predictions to {csv_path}")

In [12]:
pred = test_model(model, test_loader)
export_predictions_to_csv(model, test_loader, 'predictions.csv')

Test Loss (MSE): 0.778867
Average Displacement Error (ADE): 0.719328
Root Mean Squared Error (RMSE): 0.882535

Predicted vs Actual (First 5 examples - first ID, first pred step):
Predicted: [[2.198922  1.5997437]
 [3.4892461 2.3948143]
 [2.072135  1.5070388]
 [3.3687189 2.3262832]
 [2.3543007 1.6721517]]
Actual: [[2.0435512 1.4432989]
 [3.1072028 2.0360825]
 [1.7420435 1.314433 ]
 [3.0653267 2.0103092]
 [2.1412618 1.4690722]]
Exporting predictions to predictions.csv...


Exporting:   0%|          | 0/2 [00:00<?, ?it/s]

Finished exporting predictions to predictions.csv
